Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 4/4: Оценка качества RAG → Урок 4/4: Итоговый анализ RAG-системы: от метрик к улучшениям]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/68295d2a-b17c-4564-8957-808c64a8e401/lessons/82867876-9a3d-429f-9504-4dd8974b2073/#5e864a69-6b41-4503-8685-a5eeb4e31f18 )


In [1]:
import json

with open("knowledge_base.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]            # Список словарей: 43 раздела, поля doc_id, title, text
queries = data["queries"]      # Список словарей: 30 запросов, поля query, kind,
                               #             relevant_docs, gold

print("Разделов:", len(docs), "| Запросов:", len(queries))
print(queries[0])

# Вывод:
# Разделов: 43 | Запросов: 30
# {'query': 'что означает код ошибки E-52', 'kind': 'exact',
#  'relevant_docs': [5], 'gold': 'перегрев процессора'}

Разделов: 43 | Запросов: 30
{'query': 'что означает код ошибки E-52', 'kind': 'exact', 'relevant_docs': [5], 'gold': 'перегрев процессора'}


In [2]:
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer


def naive_split(text, size=120):
    """Разрезает text на фрагменты по size символов без перекрытия.

    Возвращает список строк; границы предложений не учитываются.
    """
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks, chunk_doc = [], []                # Тексты чанков и раздел-источник
for d in docs:
    for piece in naive_split(d["text"]):
        chunks.append(piece)
        chunk_doc.append(d["doc_id"])     # Чанк помнит, из какого раздела вырезан

print("Чанков:", len(chunks))             # Чанков: 108

# Эмбеддер темы: multilingual-e5-base с префиксами query: и passage:
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
doc_vecs = embedder.encode(["passage: " + c for c in chunks],
                           normalize_embeddings=True, show_progress_bar=False)


def search(query, k=5):
    """Ищет для query ближайшие k чанков.

    Возвращает тексты чанков и идентификаторы разделов-источников.
    """
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    top = np.argsort(q_vec @ doc_vecs.T)[::-1][:k]
    return [chunks[i] for i in top], [chunk_doc[i] for i in top]


# Генерация: тот же локальный сервер и тот же заземляющий промпт
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def generate(messages, max_tokens=200):
    """Отправляет messages локальной модели и ограничивает ответ max_tokens.

    Возвращает очищенный текст ответа; temperature=0 — для воспроизводимости.
    """
    resp = client.chat.completions.create(
        model="qwen3.5", messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()


def rag_answer(query, k=5):
    """Запускает RAG для query по k найденным чанкам.

    Возвращает ответ, тексты чанков и идентификаторы разделов-источников.
    """
    ctx, got_docs = search(query, k)
    numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(ctx, 1))
    answer = generate([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content":
         f"Фрагменты базы знаний:\n{numbered}\n\nВопрос: {query}"}])
    return answer, ctx, got_docs

Чанков: 108


In [3]:
import re

JUDGE_FAITH = (
    "Ты — строгий эксперт по оценке ответов. Оцени, насколько ОТВЕТ опирается "
    "только на ФРАГМЕНТЫ. 5 — каждое утверждение ответа есть во фрагментах; "
    "1 — ответ содержит факты, которых во фрагментах нет. Ответ «в базе знаний "
    "ответа нет» считай полностью опирающимся на фрагменты (5). "
    "Верни только одну цифру от 1 до 5.")

JUDGE_RELEV = (
    "Ты — строгий эксперт по оценке ответов. Оцени, отвечает ли ОТВЕТ по "
    "существу на ВОПРОС. 5 — прямой и полный ответ на заданный вопрос; "
    "1 — ответ не по теме или отказ отвечать. Верни только одну цифру от 1 до 5.")

JUDGE_CORRECT = (
    "Ты — строгий эксперт. Сравни ОТВЕТ с ЭТАЛОННЫМ ФАКТОМ. Верни 1, если ответ "
    "передаёт эталонный факт (формулировка может отличаться), и 0, если ответ "
    "противоречит эталону, неполон или отказывается отвечать. "
    "Верни только одну цифру: 1 или 0.")


def judge(question, context, answer, gold):
    """Оценивает answer для question по context и эталонному gold.

    Возвращает faithfulness, answer relevance и бинарную correctness.
    """
    ctx = "\n".join(f"[{i}] {c}" for i, c in enumerate(context, 1))
    faith = generate([{"role": "system", "content": JUDGE_FAITH},
                      {"role": "user", "content":
                       f"ФРАГМЕНТЫ:\n{ctx}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    relev = generate([{"role": "system", "content": JUDGE_RELEV},
                      {"role": "user", "content":
                       f"ВОПРОС: {question}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    correct = generate([{"role": "system", "content": JUDGE_CORRECT},
                        {"role": "user", "content":
                         f"ВОПРОС: {question}\nЭТАЛОННЫЙ ФАКТ: {gold}\n\n"
                         f"ОТВЕТ: {answer}"}], max_tokens=8)
    return (int(re.search(r"[1-5]", faith).group()),
            int(re.search(r"[1-5]", relev).group()),
            int(re.search(r"[01]", correct).group()))

In [5]:
import pandas as pd
# Функции rag_answer(), judge() и список queries — из общего кода

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)


def precision_at_k(got_docs, relevant):
    """Доля слотов выдачи, занятых уникальными релевантными разделами."""
    # Пропуск 1а: каждый релевантный раздел учитывайте только один раз
    # return <...>
    pass


def recall_at_k(got_docs, relevant):
    """Доля релевантных разделов, которые поиск донёс до промпта."""
    # Пропуск 1б: верните долю элементов relevant, попавших в got_docs
    # return <...>
    pass


def mrr(got_docs, relevant):
    """Обратный ранг первого релевантного раздела: 1 — первое место,
    0,5 — второе, 0 — релевантных в выдаче нет вовсе."""
    for rank, doc in enumerate(got_docs, start=1):
        # Пропуск 2: если doc релевантен, верните обратный ранг
        #<...>
        pass
    return 0.0


rows = []
for q in queries:
    answer, ctx, got_docs = rag_answer(q["query"], k=5)
    faith, relev, correct = judge(q["query"], ctx, answer, q["gold"])
    rows.append({
        "kind": q["kind"],
        "query": q["query"],
        "precision@5": precision_at_k(got_docs, q["relevant_docs"]),
        "recall@5": recall_at_k(got_docs, q["relevant_docs"]),
        "MRR": mrr(got_docs, q["relevant_docs"]),
        "faithfulness": faith,
        "answer relevance": relev,
        "correctness": correct,
        "answer": answer,
        "ctx": ctx,
    })

report = pd.DataFrame(rows)
METRICS = ["precision@5", "recall@5", "MRR", "faithfulness",
           "answer relevance", "correctness"]

# Пропуск 3: выведите средние по всему набору и разбивку по типам запросов
print("Сводный профиль:")
# print(<...>)
print("\nПо типам запросов:")
# print(<...>)

ModuleNotFoundError: No module named 'pandas'